In [1]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

dataset_name = "irb_new"
config = "2_citations"

work_dir = f"/home/lamdo/splade/data/{dataset_name}"
qa_metadata_folder = "/scratch/lamdo/IRB/qa_metadata"
retrieval_metadata_folder = "/scratch/lamdo/IRB/retrieval_metadata"
experiment_name_with_retrieval = f"{dataset_name}__bm25__rc1" if not config else f"{dataset_name}_{config}__bm25__rc1"
experiment_name_without_retrieval = f"{dataset_name}__bm25__rc0" if not config else f"{dataset_name}_{config}__bm25__rc0"


queries_file = os.path.join(work_dir, "queries.jsonl") if not config else os.path.join(work_dir, config, "queries.jsonl")

groundtruth_file_with_retrieval = os.path.join(qa_metadata_folder, f"{experiment_name_with_retrieval}.refs.txt")
prediction_file_with_retrieval = os.path.join(qa_metadata_folder, f"{experiment_name_with_retrieval}.hyps.txt")
evaluation_metadata_file_with_retrieval = os.path.join(qa_metadata_folder, f"{experiment_name_with_retrieval}.evaluation_metadata.txt")


groundtruth_file_without_retrieval = os.path.join(qa_metadata_folder, f"{experiment_name_without_retrieval}.refs.txt")
prediction_file_without_retrieval = os.path.join(qa_metadata_folder, f"{experiment_name_without_retrieval}.hyps.txt")
evaluation_metadata_file_without_retrieval = os.path.join(qa_metadata_folder, f"{experiment_name_without_retrieval}.evaluation_metadata.txt")


retrieval_contexts_file = os.path.join(retrieval_metadata_folder, f"{dataset_name}__bm25.json")

In [2]:
with open(retrieval_contexts_file) as f:
    retrieval_contexts = json.load(f)

In [3]:
queries = []
queries_id = []
with open(queries_file) as f:
    for line in f:
        jline = json.loads(line)
        queries.append(jline["text"])
        queries_id.append(jline["_id"])

In [4]:
predictions_with_retrieval = []
groundtruths_with_retrieval = []
eval_metadata_with_retrieval = []

with open(groundtruth_file_with_retrieval) as f:
    for line in f:
        groundtruths_with_retrieval.append(line)

with open(prediction_file_with_retrieval) as f:
    for line in f:
        predictions_with_retrieval.append(line)

with open(evaluation_metadata_file_with_retrieval) as f:
    for line in f:
        eval_metadata_with_retrieval.append(line)



predictions_without_retrieval = []
groundtruths_without_retrieval = []
eval_metadata_without_retrieval = []

with open(groundtruth_file_without_retrieval) as f:
    for line in f:
        groundtruths_without_retrieval.append(line)

with open(prediction_file_without_retrieval) as f:
    for line in f:
        predictions_without_retrieval.append(line)

with open(evaluation_metadata_file_without_retrieval) as f:
    for line in f:
        eval_metadata_without_retrieval.append(line)

In [5]:
# check to see if retrieval or no retrieval is more beneficial


# Your original matrix filling code here
matrix = np.zeros([3, 3])
evaluation_differences = {}
for i, (with_retrieval, without_retrieval) in enumerate(zip(eval_metadata_with_retrieval, eval_metadata_without_retrieval)):
    with_retrieval_index = int(with_retrieval) + 1
    without_retrieval_index = int(without_retrieval) + 1
    matrix[without_retrieval_index][with_retrieval_index] += 1

    from_l = ['INCORRECT', 'MISSING', 'CORRECT'][without_retrieval_index]
    to_l = ['INCORRECT', 'MISSING', 'CORRECT'][with_retrieval_index]

    key = f"{from_l}->{to_l}"
    if key not in evaluation_differences:
        evaluation_differences[key] = []
    
    to_append = {
        "query": queries[i],
        "query_id": queries_id[i],
        "prediction_without_retrieval": predictions_without_retrieval[i],
        "prediction_with_retrieval": predictions_with_retrieval[i],
        "retrieval_contexts": "\n\n#####CONTEXT SEPARATE LINE#####\n\n".join([item["contents"] for item in retrieval_contexts[queries_id[i]][:5]]),
        "groundtruth": groundtruths_with_retrieval[i]
    }
    evaluation_differences[key].append(to_append)



matrix = matrix / np.sum(matrix)

# Create an empty RGB array
color_matrix = np.zeros(matrix.shape + (3,))

# Normalize values for coloring
max_val = np.max(matrix)
if max_val == 0: 
    max_val = 1  # prevent division by zero

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val_norm = matrix[i, j] / max_val  # normalized 0 to 1

        if i == j:
            # Diagonal gray (medium gray)
            color_matrix[i, j] = [0.5, 0.5, 0.5]
        elif j > i:
            # Upper right: white to green transition
            # Interpolate between white and green
            color_matrix[i, j] = [1 - val_norm, 1, 1 - val_norm]  # Red and Blue channels fade from 1 to 0, Green stays 1
        else:
            # Lower left: white to red transition
            # Interpolate between white and red
            color_matrix[i, j] = [1, 1 - val_norm, 1 - val_norm]  # Green and Blue fade from 1 to 0, Red stays 1

fig, ax = plt.subplots()
cax = ax.imshow(color_matrix)

# Set integer ticks and labels
ax.set_xticks([0, 1, 2])
ax.set_yticks([0, 1, 2])
ax.set_xticklabels(['INCORRECT', 'MISSING', 'CORRECT'])
ax.set_yticklabels(['INCORRECT', 'MISSING', 'CORRECT'])
ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

# X label on top
ax.xaxis.set_label_position('top')
plt.xlabel("With Retrieval")
plt.ylabel("Without Retrieval")

# Add percentage text annotations
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        text_color = 'white' if (color_matrix[i, j].sum() < 1.5) else 'black'  # adjust text color for readability
        ax.text(j, i, f"{matrix[i, j] * 100:.1f}%", ha='center', va='center', color=text_color)

plt.show()


KeyError: '2025 New Orleans truck attack--55'

In [25]:
INDEX_TO_INSPECT = 0
INSPECT_TYPE = "CORRECT->MISSING"
to_inspect = evaluation_differences[INSPECT_TYPE][INDEX_TO_INSPECT]

print(f"ID: {to_inspect['query_id']}\n\nQ: {to_inspect['query']}\n\nGT: {to_inspect['groundtruth']}\n\nP (w/o retrieval): {to_inspect['prediction_without_retrieval']}\n\nP (w retrieval): {to_inspect['prediction_with_retrieval']}\n\nRC: {to_inspect['retrieval_contexts']}")

ID: Face transplant--22

Q: Who performed the world's first partial face transplant on a living human and when was it done?

GT: The world's first partial face transplant on a living human was performed on 27 November 2005 by Bernard Devauchelle, a French oral and maxillofacial surgeon, Benoit Lengele, a Belgian plastic surgeon, and Jean-Michel Dubernard in Amiens, France.


P (w/o retrieval): The world's first partial face transplant on a living human was performed by Dr. Bernard Devauchelle and his team in France on November 27, 2005.


P (w retrieval): French surgeons performed the world's first partial face transplant on a living human in 2005.


RC: Shrek was the best that even the most skilled reconstructive plastic surgeons could do for a patient as severely injured as she was. Katie would have lived the rest of her life concealing what she could of her face with surgical masks and scarves, hearing the startled whispers of strangers when she went out in public, and struggling to

In [7]:
predictions_with_retrieval[:2]

['Hercule Poirot is a meticulous master detective created by novelist Agatha Christie.\n',
 "The origin of Hercule Poirot's name is not explicitly mentioned in the provided contexts.\n"]

In [8]:
matrix

array([[0.14, 0.15, 0.13],
       [0.04, 0.18, 0.09],
       [0.02, 0.12, 0.13]])

In [9]:
predictions_without_retrieval[:2]

['Hercule Poirot is a fictional Belgian detective created by British author Agatha Christie.\n',
 'Hercule Poirot\'s name was created by Agatha Christie. The name "Hercule" is derived from the Greek hero Hercules, symbolizing strength and heroism, while "Poirot" is a common Belgian surname, reflecting his Belgian nationality.\n']

In [6]:
variation = "without"

predictions_to_view = {
    "with": predictions_with_retrieval,
    "without": predictions_without_retrieval
}

for idx, (query, pred, gt) in enumerate(zip(queries, predictions_to_view[variation], groundtruths_with_retrieval)):
    eval_label = eval_metadata_with_retrieval[idx].strip("\n")
    eval_label_text = eval_label
    print(f"{eval_label_text}\nQ: {query.strip()}\nP: {pred.strip()}\nGT: {gt.strip()}")
    print()

1.0,0.0,0.0
Q: Who is the fictional Belgian detective created by the English writer Agatha Christie?
P: Hercule Poirot.
GT: Hercule Poirot is a fictional Belgian detective created by the English writer Agatha Christie.

1.0,0.0,0.0
Q: Did David Suchet, the actor who portrayed Hercule Poirot on television, describe the character as obsessive-compulsive?
P: Yes, David Suchet did describe Hercule Poirot as obsessive-compulsive.
GT: The actor David Suchet, who portrayed Hercule Poirot on television, stated that Poirot is obsessive-compulsive.

0.0,0.0,1.0
Q: What filmmaker noted that he enjoyed portraying the obsessive-compulsive traits of Hercule Poirot?
P: Kenneth Branagh
GT: The filmmaker Kenneth Branagh stated that he enjoyed portraying the obsessive-compulsive traits of Hercule Poirot.

1.0,0.0,0.0
Q: Who will star as Hercule Poirot in the new touring production of a Poirot play?
P: Ben Nealon will star as Hercule Poirot in the new touring production of a Poirot play.
GT: A new tourin

In [186]:
# with retrieval
print("with retrieval")
counter = Counter([item.strip("\n") for item in eval_metadata_with_retrieval])
for l, l_text in zip(["-1", "0", "1"], ["INCORRECT", "MISSING", "CORRECT"]):
    print(l_text, counter[l] / sum(counter.values()))

print()
# without retrieval
print("withOUT retrieval")
counter = Counter([item.strip("\n") for item in eval_metadata_without_retrieval])
for l, l_text in zip(["-1", "0", "1"], ["INCORRECT", "MISSING", "CORRECT"]):
    print(l_text, counter[l] / sum(counter.values()))

with retrieval
INCORRECT 0.31
MISSING 0.45
CORRECT 0.24

withOUT retrieval
INCORRECT 0.58
MISSING 0.24
CORRECT 0.18


In [138]:
float('0.5\n')

0.5